# OpenPlaque — RCA PCAT outer-wall sensitivity analysis

This notebook asks one focused question:

> **Does the RCA 10–50 mm PCAT result remain stable when the approximate outer-wall offset is varied?**

It holds the **centerline and lumen-radius profile fixed** from the accepted PCAT prototype and reruns only the perivascular sampling geometry for outer-wall margins:

**0.25, 0.50, 0.75, 1.00, 1.25 mm**

For each margin it reports:
- overall OpenPlaque PCAT attenuation
- adipose voxel count and volume
- near-wall (0–2 mm outward) attenuation
- outer-layer (4–6 mm outward) attenuation
- outer-minus-near difference
- radial slope over the well-supported 0–5 mm layers
- longitudinal 1-mm profile

This is a research prototype, **not Caristo FAI-Score**.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-wall-sensitivity-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas
print('Repository and packages ready.')


In [ ]:
import sys, shutil, math, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
BASE = ROOT/'PCAT_RCA_10_50'
OUT = ROOT/'PCAT_RCA_10_50_Wall_Sensitivity'
OUT.mkdir(parents=True, exist_ok=True)

MARGINS_MM = [0.25, 0.50, 0.75, 1.00, 1.25]
FAT_LO_HU, FAT_HI_HU = -190.0, -30.0
SEGMENT_START_MM, SEGMENT_END_MM = 10.0, 50.0

print('Output:', OUT)


## 1. Load source CCTA and freeze the accepted coordinate/radius model

The sensitivity analysis **does not recalculate the centerline or lumen radius**. Those are treated as fixed inputs so the only variable is the assumed wall margin.


In [ ]:
DRIVE_ZIP = ROOT/'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_pcat_wall_sensitivity'

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
source_img, ct, _ = study.load_series(7)
ct = np.asarray(ct)
sp_xyz = np.array(source_img.GetSpacing(), float)
sp_zyx = sp_xyz[::-1]
voxel_mm3 = float(np.prod(sp_xyz))

def first_existing(names):
    for p in names:
        if p.exists():
            return p
    return None

centerline_path = first_existing([
    BASE/'rca_centerline_smoothed_zyx.csv',
    *ROOT.rglob('rca_centerline_smoothed_zyx.csv')
])
radius_path = first_existing([
    BASE/'pcat_local_radius_profile.csv',
    *ROOT.rglob('pcat_local_radius_profile.csv')
])

if centerline_path is None or radius_path is None:
    raise FileNotFoundError('Run the RCA 10–50 mm PCAT prototype first; fixed centerline/radius files were not found.')

cl = pd.read_csv(centerline_path)
rad = pd.read_csv(radius_path)

need_cl = {'z','y','x','arc_mm'}
need_rad = {'arc_mm','lumen_radius_mm'}
if not need_cl.issubset(cl.columns):
    raise ValueError(f'Centerline columns missing. Need {need_cl}; got {list(cl.columns)}')
if not need_rad.issubset(rad.columns):
    raise ValueError(f'Radius columns missing. Need {need_rad}; got {list(rad.columns)}')

lumen_r_all = np.interp(cl.arc_mm.to_numpy(float),
                        rad.arc_mm.to_numpy(float),
                        rad.lumen_radius_mm.to_numpy(float))

seg_mask = (cl.arc_mm.to_numpy(float) >= SEGMENT_START_MM) & (cl.arc_mm.to_numpy(float) <= SEGMENT_END_MM)
seg = cl.loc[seg_mask].reset_index(drop=True)
seg_arc = seg.arc_mm.to_numpy(float)
seg_zyx = seg[['z','y','x']].to_numpy(float)
seg_mm = seg_zyx * sp_zyx
seg_lumen_r = lumen_r_all[seg_mask]

print('CT shape:', ct.shape)
print('Spacing xyz mm:', tuple(sp_xyz))
print('Fixed centerline:', centerline_path)
print('Fixed radius profile:', radius_path)
print('Segment points:', len(seg), 'arc:', float(seg_arc.min()), 'to', float(seg_arc.max()))
print('Mean fixed lumen radius mm:', round(float(seg_lumen_r.mean()), 3))


## 2. Build one common physical crop and nearest-centerline map

The same voxel set and same nearest-centerline assignment are reused for **every margin**. This makes the sensitivity comparison clean.


In [ ]:
max_outer_r = float(np.max(seg_lumen_r + max(MARGINS_MM)))
max_shell_outer = 3.0 * max_outer_r
pad_mm = max_shell_outer + 3.0

lo = np.floor(np.min(seg_zyx, axis=0) - pad_mm/sp_zyx).astype(int)
hi = np.ceil(np.max(seg_zyx, axis=0) + pad_mm/sp_zyx).astype(int) + 1
lo = np.maximum(lo, 0)
hi = np.minimum(hi, np.array(ct.shape))

crop = ct[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
zz,yy,xx = np.indices(crop.shape)
glob_zyx = np.stack([zz+lo[0], yy+lo[1], xx+lo[2]], axis=-1).reshape(-1,3).astype(float)
glob_mm = glob_zyx * sp_zyx

tree = cKDTree(seg_mm)
dist_mm, nearest_idx = tree.query(glob_mm, k=1, workers=-1)
nearest_idx = nearest_idx.astype(int)
nearest_arc = seg_arc[nearest_idx]
nearest_lumen = seg_lumen_r[nearest_idx]
hu_flat = crop.reshape(-1).astype(float)
fat_hu = (hu_flat >= FAT_LO_HU) & (hu_flat <= FAT_HI_HU)

aorta_candidates = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz',
]
ap = next((p for p in aorta_candidates if p.exists()), None)
if ap is not None:
    ai = sitk.ReadImage(str(ap))
    if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(), source_img.GetSpacing()):
        ai = sitk.Resample(ai, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
    aorta = sitk.GetArrayFromImage(ai)>0
    aorta_crop = aorta[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]].reshape(-1)
else:
    aorta_crop = np.zeros_like(hu_flat, dtype=bool)

print('Crop zyx:', crop.shape)
print('Crop voxels:', len(hu_flat))
print('Max modeled shell radius mm:', round(max_shell_outer,2))
print('Aorta exclusion available:', ap is not None)


## 3. Recompute PCAT for each outer-wall margin


In [ ]:
summary_rows = []
radial_rows = []
long_rows = []
masks_for_qc = {}

for margin in MARGINS_MM:
    outer = nearest_lumen + margin
    shell_outer = 3.0 * outer
    radial_outward = dist_mm - outer

    shell = (
        (dist_mm > outer) &
        (dist_mm <= shell_outer) &
        (nearest_arc >= SEGMENT_START_MM) &
        (nearest_arc <= SEGMENT_END_MM) &
        (~aorta_crop)
    )
    fat = shell & fat_hu

    vals = hu_flat[fat]
    if len(vals) == 0:
        raise RuntimeError(f'No fat voxels for margin {margin}')

    rmeans = []
    rcounts = []
    for b in range(0, 7):
        m = fat & (radial_outward >= b) & (radial_outward < b+1)
        vv = hu_flat[m]
        radial_rows.append({
            'wall_margin_mm': margin,
            'radial_bin_start_mm': float(b),
            'radial_bin_end_mm': float(b+1),
            'fat_voxels': int(len(vv)),
            'fat_volume_ml': float(len(vv)*voxel_mm3/1000.0),
            'mean_hu': float(np.mean(vv)) if len(vv) else np.nan,
            'median_hu': float(np.median(vv)) if len(vv) else np.nan,
        })
        rmeans.append(float(np.mean(vv)) if len(vv) else np.nan)
        rcounts.append(int(len(vv)))

    mids = np.arange(0.5,5.0,1.0)
    first5 = np.array(rmeans[:5], float)
    first5_counts = np.array(rcounts[:5], int)
    if np.all(np.isfinite(first5)) and np.min(first5_counts) >= 100:
        slope = float(np.polyfit(mids, first5, 1)[0])
    else:
        slope = np.nan

    near = fat & (radial_outward >= 0) & (radial_outward < 2)
    outer_layer = fat & (radial_outward >= 4) & (radial_outward < 6)
    near_vals = hu_flat[near]
    outer_vals = hu_flat[outer_layer]
    near_mean = float(np.mean(near_vals)) if len(near_vals) else np.nan
    outer_mean = float(np.mean(outer_vals)) if len(outer_vals) else np.nan
    delta = outer_mean-near_mean if np.isfinite(near_mean) and np.isfinite(outer_mean) else np.nan

    for b in range(int(SEGMENT_START_MM), int(SEGMENT_END_MM)):
        m = fat & (nearest_arc >= b) & (nearest_arc < b+1)
        vv = hu_flat[m]
        long_rows.append({
            'wall_margin_mm': margin,
            'arc_bin_start_mm': float(b),
            'arc_bin_end_mm': float(b+1),
            'fat_voxels': int(len(vv)),
            'fat_volume_ml': float(len(vv)*voxel_mm3/1000.0),
            'mean_hu': float(np.mean(vv)) if len(vv) else np.nan,
        })

    summary_rows.append({
        'wall_margin_mm': margin,
        'pcat_mean_hu': float(np.mean(vals)),
        'pcat_median_hu': float(np.median(vals)),
        'pcat_sd_hu': float(np.std(vals)),
        'fat_voxels': int(len(vals)),
        'fat_volume_ml': float(len(vals)*voxel_mm3/1000.0),
        'shell_voxels': int(np.sum(shell)),
        'shell_volume_ml': float(np.sum(shell)*voxel_mm3/1000.0),
        'fat_fraction': float(len(vals)/max(1,np.sum(shell))),
        'near_0_2mm_mean_hu': near_mean,
        'near_0_2mm_voxels': int(len(near_vals)),
        'outer_4_6mm_mean_hu': outer_mean,
        'outer_4_6mm_voxels': int(len(outer_vals)),
        'outer_minus_near_hu': delta,
        'radial_slope_0_5_hu_per_mm': slope,
        'min_radial_bin_count_0_5': int(np.min(first5_counts)),
    })

    masks_for_qc[margin] = {
        'shell': shell.reshape(crop.shape),
        'fat': fat.reshape(crop.shape),
    }

summary = pd.DataFrame(summary_rows)
radial = pd.DataFrame(radial_rows)
longitudinal = pd.DataFrame(long_rows)

summary.to_csv(OUT/'pcat_wall_sensitivity_summary.csv', index=False)
radial.to_csv(OUT/'pcat_wall_sensitivity_radial.csv', index=False)
longitudinal.to_csv(OUT/'pcat_wall_sensitivity_longitudinal.csv', index=False)

display(summary)


## 4. Stability metrics


In [ ]:
mean_range = float(summary.pcat_mean_hu.max() - summary.pcat_mean_hu.min())
delta_signs = np.sign(summary.outer_minus_near_hu.dropna().to_numpy(float))
slope_signs = np.sign(summary.radial_slope_0_5_hu_per_mm.dropna().to_numpy(float))
gradient_consistent = bool(len(delta_signs)==len(MARGINS_MM) and np.all(delta_signs == delta_signs[0]))
slope_consistent = bool(len(slope_signs)==len(MARGINS_MM) and np.all(slope_signs == slope_signs[0]))

stability = pd.DataFrame([{
    'pcat_mean_range_across_margins_hu': mean_range,
    'pcat_mean_sd_across_margins_hu': float(summary.pcat_mean_hu.std(ddof=0)),
    'outer_minus_near_sign_consistent': gradient_consistent,
    'radial_slope_sign_consistent': slope_consistent,
    'all_margins_outer_minus_near_positive': bool(np.all(summary.outer_minus_near_hu > 0)),
    'all_margins_radial_slope_positive': bool(np.all(summary.radial_slope_0_5_hu_per_mm > 0)),
}])
stability.to_csv(OUT/'pcat_wall_sensitivity_stability.csv', index=False)
display(stability.T)


## 5. Sensitivity figures


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(summary.wall_margin_mm, summary.pcat_mean_hu, marker='o', label='overall PCAT mean')
ax.plot(summary.wall_margin_mm, summary.near_0_2mm_mean_hu, marker='o', label='near wall 0–2 mm')
ax.plot(summary.wall_margin_mm, summary.outer_4_6mm_mean_hu, marker='o', label='outer 4–6 mm')
ax.set_xlabel('assumed wall margin (mm)')
ax.set_ylabel('mean HU')
ax.set_title('PCAT sensitivity to outer-wall offset')
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
p1=OUT/'01_wall_sensitivity_summary.png'
fig.savefig(p1,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for margin in MARGINS_MM:
    q = radial[radial.wall_margin_mm==margin]
    x = (q.radial_bin_start_mm + q.radial_bin_end_mm)/2
    ax.plot(x, q.mean_hu, marker='o', label=f'+{margin:.2f} mm')
ax.set_xlabel('mm outward from modeled outer wall')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Radial PCAT profiles by wall assumption')
ax.legend(title='wall margin')
ax.grid(alpha=.25)
plt.tight_layout()
p2=OUT/'02_radial_profiles_by_margin.png'
fig.savefig(p2,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for margin in MARGINS_MM:
    q=longitudinal[longitudinal.wall_margin_mm==margin]
    x=(q.arc_bin_start_mm+q.arc_bin_end_mm)/2
    ax.plot(x,q.mean_hu,label=f'+{margin:.2f} mm')
ax.set_xlabel('arc length from ostium (mm)')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Longitudinal PCAT profiles by wall assumption')
ax.legend(title='wall margin',ncol=2)
ax.grid(alpha=.25)
plt.tight_layout()
p3=OUT/'03_longitudinal_profiles_by_margin.png'
fig.savefig(p3,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 6. Cross-sectional QC at the two extreme wall assumptions

These panels make it easy to see how much the actual sampled fat region changes between +0.25 mm and +1.25 mm.


In [ ]:
targets=[10,20,30,40,50]
fig, axs = plt.subplots(2,5,figsize=(18,7))

for row, margin in enumerate([min(MARGINS_MM), max(MARGINS_MM)]):
    fat_crop = masks_for_qc[margin]['fat']
    shell_crop = masks_for_qc[margin]['shell']
    for col,targ in enumerate(targets):
        j=int(np.argmin(np.abs(seg_arc-targ)))
        gz,gy,gx=np.round(seg_zyx[j]).astype(int)
        cz=gz-lo[0]
        r=35
        y0=max(0,gy-r); y1=min(ct.shape[1],gy+r)
        x0=max(0,gx-r); x1=min(ct.shape[2],gx+r)
        ax=axs[row,col]
        ax.imshow(ct[gz,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=600)
        if 0 <= cz < fat_crop.shape[0]:
            sy0=max(0,y0-lo[1]); sy1=min(fat_crop.shape[1],y1-lo[1])
            sx0=max(0,x0-lo[2]); sx1=min(fat_crop.shape[2],x1-lo[2])
            m_shell=np.zeros((y1-y0,x1-x0),bool)
            m_fat=np.zeros_like(m_shell)
            dy0=(lo[1]+sy0)-y0; dx0=(lo[2]+sx0)-x0
            sub_shell=shell_crop[cz,sy0:sy1,sx0:sx1]
            sub_fat=fat_crop[cz,sy0:sy1,sx0:sx1]
            m_shell[dy0:dy0+sub_shell.shape[0],dx0:dx0+sub_shell.shape[1]]=sub_shell
            m_fat[dy0:dy0+sub_fat.shape[0],dx0:dx0+sub_fat.shape[1]]=sub_fat
            if np.any(m_shell): ax.contour(m_shell.astype(float),levels=[.5],linewidths=1)
            yy,xx=np.where(m_fat)
            if len(xx): ax.scatter(xx,yy,s=3,alpha=.35)
        ax.plot(gx-x0,gy-y0,'o',markersize=4)
        ax.set_title(f'{targ} mm, wall +{margin:.2f}')
        ax.axis('off')

plt.tight_layout()
p4=OUT/'04_extreme_margin_qc.png'
fig.savefig(p4,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 7. Package everything for report-back

The ZIP is the only file you need to send back in ChatGPT. The notebook also prints direct Google Drive search URLs for every output.


In [ ]:
files_to_package = [
    p1,p2,p3,p4,
    OUT/'pcat_wall_sensitivity_summary.csv',
    OUT/'pcat_wall_sensitivity_radial.csv',
    OUT/'pcat_wall_sensitivity_longitudinal.csv',
    OUT/'pcat_wall_sensitivity_stability.csv',
]

zip_path = OUT/'PCAT_WALL_SENSITIVITY_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_package:
        if p.exists():
            zf.write(p,arcname=p.name)

url_rows=[]
for p in [zip_path]+files_to_package:
    q=p.name.replace(' ','%20')
    url=f'https://drive.google.com/drive/u/0/search?q={q}'
    url_rows.append({'file':p.name,'drive_search_url':url})

urls=pd.DataFrame(url_rows)
urls.to_csv(OUT/'REPORT_BACK_URLS.csv',index=False)

print('\n=== REPORT BACK ===')
print('Send just this ZIP:')
print(zip_path)
print('\nDirect Drive search URL:')
print(f'https://drive.google.com/drive/u/0/search?q={zip_path.name}')
print('\nAll output URLs:')
for _,r in urls.iterrows():
    print(f"{r['file']}: {r['drive_search_url']}")

print('\nSensitivity analysis complete.')
